## Overview

Read the original Excel workbook that came from CMS for the 3070G forms, 10-year history. 

* Shorten some column names
* Standardize the types of ownership

The staffing numbers are a problem that are handled elsewhere. 

In [15]:
pacman::p_load(
  "tidyverse",
  "lubridate",
  "janitor",
  "googledrive",
  "readxl",
  "here"
)


suppressMessages(here::i_am("cms_3070g/01-read-cms-3070g.ipynb" ))
# I'm never sure about paths - just making sure here.
setwd(here())
getwd()



[1] "/Users/sarah/Github/nc-homesdisabled"

Original download from the shared google drive: `tmp <- drive_download("https://drive.google.com/file/d/1piW8fXmuDYtNxNoaqcXi7uLYkMJSfQ7P", "cms_formdata.xlsx", overwrite=TRUE)`


Read the names from the appropriate line in the Excel sheet, then use them to import. Changes made: 

1. For fields that begin with "num" in the name, convert NA to 0. 
2. Then convert age to numeric after that. 
3. Convert the survey date
4. Add missing ccns - I could only find one. 
5. Rename columns to remove "num_", "clients_" and "parent_organization", just making them a little shorter. 
6. create a resident estimate based on total of male + female. It's always the same as the one shown, but I'm keeping it because later programs depend on having that variable.
7. create iniital staff ratio - this will be fixed in another script but this is the original. Use the entire staff not just the direct care staff for this ratio - I wasn't clear from the regs which one should be used, and the people might be different. (Example: they use nurses for direct care instead, wouldn't be counted.)

This produces warnings because a number of the numeric fields in Excel are stored as character. It's OK. 



In [16]:

nms <- names(read_excel("cms_3070g/cms_formdata.xlsx"))


data_types <-
  case_when(
    str_starts(nms, "num") ~ "numeric",
    #               date parsing didn't work so do it later.
    #               str_detect(nms, "Date") ~ "date",
    .default = "text"
  )



cms_data <- read_excel("cms_3070g/cms_formdata.xlsx", col_types = data_types) |>
  clean_names() |>
   # for numeric fields, replace NA with 0 -- That's really the only thing we can do.
  # I didn't see hardly any 0's in the original data.

  mutate(across(where(is.numeric), \(x) replace_na(x, 0))) |>

  mutate(across(contains("_age"), as.numeric)) |>

  mutate(survey_date = ymd(survey_exit_date), .before = survey_exit_date) |>


  # missing ccns. Could only find one of the two
  mutate(
    provider_ccn = provider_ccn |>
      replace_when(name_of_facility == "Floral Park Care" ~ "55G829"), 
    resident_estimate =  num_clients_male + num_clients_female, .after=num_icfiid_beds
  ) |> 
  rename_with ( \(x) str_remove (x, "num_") , starts_with("num_") ) |> 
  rename_with ( \(x) str_replace (x, "larger_organization", "parent"))  |> 
  rename_with ( \(x) str_remove (x, "clients_"), starts_with("clients_") )  |> 
  mutate( staff_ratio = total_personnel / resident_estimate, .after=total_personnel)

# the warnings come because there are several fields that were defined as characgter in the excel file, but are all numbers.

There were 17 warnings (use warnings() to see them)
There were 50 or more warnings (use warnings() to see the first 50)


There was some confusion over what the correct number of clients should be. In the Stanford paper, they thought there was some confusion between the number of beds being for the parent or the facility. That seems true for the beds in the larger organization, but not in the icf beds. The cases where the icf beds were really high compared with residents were almost all in the large state facilities that are shrinking really fast. So I'm keeping both but using residents for most of the ratios.

Check the totals and see how far off they are.


In [10]:
cms_data |> 
  filter ( resident_estimate != total_clients_by_sex )  |> 
  select ( provider_ccn, name_of_facility, state_code, icfiid_beds, resident_estimate, total_clients_by_age, total_clients_by_sex)

# A tibble: 1 × 7
  provider_ccn name_of_facility  state_code icfiid_beds resident_estimate total_clients_by_age total_clients_by_sex
  <chr>        <chr>             <chr>            <dbl>             <dbl>                <dbl>                <dbl>
1 45G313       SPRING GROUP HOME TX                   6                 5                    6                    6

Check icf beds vs. resident estimate

In [12]:
cms_data |>  filter_out ( between ( icfiid_beds/resident_estimate, .9, 1.2 ) ) |>  select ( provider_ccn, name_of_facility, state_code, beds_parent, icfiid_beds, resident_estimate) 

# A tibble: 7,603 × 6
   provider_ccn name_of_facility  state_code beds_parent icfiid_beds resident_estimate
   <chr>        <chr>             <chr>            <dbl>       <dbl>             <dbl>
 1 55G829       Floral Park Care  CA                   0           0                 3
 2 NA           Jane Bansil Olano CA                   0           0                 1
 3 03G001       ATP - WINDSOR     AZ                   0          12                 9
 4 03G001       ATP - WINDSOR     AZ                   0          12                 7
 5 03G001       ATP - WINDSOR     AZ                  12          12                 8
 6 03G001       ATP - WINDSOR     AZ                   0          12                 6
 7 03G001       ATP - WINDSOR     AZ                  12          12                 7
 8 03G001       ATP - WINDSOR     AZ                  12          12                 6
 9 03G001       ATP - WINDSOR     AZ                  12          12                 8
10 03G001       ATP -

From what I can see, the icfiid beds are correct (or as correct as anything else), while the parent beds is kind of random. The places where it's way off of the resident count are the huge state facilities that have been shrinking very fast. If they're like NC, then they stay licensed but not in use.

## Standardize ownership types

The ownership types are not at all standardized. This creates a table of unique original types and recodes them. 
Originally, I'd used some regex (commented out below) to set them, but they really weren't very good. Then I got Claude Code to do it for me, which used a bunch of tokens and didn't do much better (saved here as `classify_owners.R`). Then I just used OpenRefine, which worked fine, saved as `owners-csv-openrefine.tsv`. 


In [ ]:
ownership_types <- 
  cms_data |>
  count(type_of_ownership) |>
  arrange(desc(n)) 

# cleaned_ownership <- ownership_types |>
#   mutate(
#     clean_type = str_to_upper(type_of_ownership),
#     nyt_owner_type = case_when(
#       # --- Rule 1: Non-Profit ---
#       # do this first to get the quasi-non-profits here.
#       str_detect(
#         clean_type,
#         "NON[- ]*PROF(IT)?|NOT FOR PROFIT|CHARITABLE|DSN BOARD|DSNB|QUASI"
#       ) ~ "Non-profit",

#       # --- Rule 2: Government ---
#       str_detect(
#         clean_type,
#         "GOV|STATE|COUNTY|CITY|LOCAL|POLITICAL|MHMR|CSB|LIDDA|COMMUNITY"
#       ) ~ "Government",

#       # --- Rule 3: Private (proprietary) / For-Profit ---
#       str_detect(
#         clean_type,
#         "PROPRIETARY|PROFIT|LLC|LIMITED LIAB|PARTNERSHIP|S-?CORP|C-?CORP|SOLE PROP|PUBLICLY TRADED|CORP|INC"
#       ) ~ "Investor owned",

#       # --- Rule 4: Other / Unclassified ---
#       .default = "Other"
#     )
#   ) |>
#   # Select only the original column and the new final category for the output
#   select(type_of_ownership, nyt_owner_type)




In [20]:
ownership_types_edited <- read_tsv("cms_3070g/owners-csv-openrefine.tsv", na=c("", "NA") , show_col_types = FALSE) |> 
  select ( type_of_ownership, nyt_owner_type = translated_owner_type)

## Saved data

Join with ownership / shorten some field names

In [ ]:

all_years_all_fields <-
  cms_data  |>
  left_join(ownership_types_edited, join_by(type_of_ownership)) |>
  relocate ( nyt_owner_type , .after=type_of_ownership) |> 
  arrange ( provider_ccn , survey_date)


  
## Uncomment to save again
# saveRDS(all_years_all_fields, here("cms_3070g/cms3070g_allfields.RDS"))

glimpse(all_years_all_fields)


Rows: 48,969
Columns: 73
$ provider_ccn                            <chr> "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G012", "01G012", "01G012", "01G012", "01G012", "01G012", "01G012", "…
$ name_of_facility                        <chr> "DAVIS / ORMAN INTERMEDIATE CARE FACILITY", "DAVIS / ORMAN INTERMEDIATE CARE FACILITY", "DAVIS / ORMAN INTERMEDIATE CARE FACILITY", "DAVIS / ORMAN INTE…
$ address_line1                           <chr> "202 HOPE DRIVE", "202 HOPE DRIVE", "202 HOPE DRIVE", "202 HOPE DRIVE", "202 HOPE DRIVE", "202 HOPE DRIVE", "202 HOPE DRIVE", "202 HOPE DRIVE", "101 HO…
$ address_line2                           <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ city                                    <chr> "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTABOGA", "EASTA

#### Create 5-year summary



In [ ]:

five_year_summary <-
  all_years_all_fields |>
  arrange(provider_ccn, survey_date) |>
  group_by(provider_ccn) |>
  slice_tail(n = 5) |>
  mutate(
    survey_years = n(),
    first_survey = first(survey_date),
    median_client_est = median(resident_estimate),
    allegations = sum(total_allegations_investigated),
    deaths = sum(total_deaths),
    median_restraints = median(physically_restrained),
    deaths_suspect = sum(deaths_restraints + deaths_unusual),
    median_direct_care = median(direct_care_personnel),
    median_staff = median(total_personnel), 
    median_ratio = median(staff_ratio),
    median_sev_prof_id = median(
      severe_intellectual_disability + profound_intellectual_disability 
    )
  ) |>
  slice_tail(n = 1) |>
  ungroup() |>
  mutate(
    fac_group = case_when(
      str_detect(parent_name, "RHA ") ~ "RHA",
      str_detect(nyt_owner_type, "Investor-owned") ~ "For profit",
      nyt_owner_type == "Non-profit" ~ "Non-profit",
      nyt_owner_type == "State" ~ "State",
      .default = "Other"
    ) |>
      fct_relevel("RHA", "For profit", "Non-profit", "State", "Other")
  ) |>
  select(
    provider_ccn,
    survey_years,
    name_of_facility,
    city,
    state_code,
    nyt_owner_type,
    orig_owner_type = type_of_ownership,
    fac_group,
    parent_name,
    parent_zip,
    median_client_est,
    median_direct_care,
    oldest_client_age,
    youngest_client_age,
    median_sev_prof_id,
    deaths,
    deaths_suspect,
    allegations,
    first_survey,
    last_survey = survey_date
  ) 


# str(five_year_summary$fac_group)

# uncomment to resave
# saveRDS(five_year_summary, "cms_3070g/five_year_summary.rds")